# Mencoba model

## Import library

In [78]:
import re
import string
import json
import emoji
import joblib
import requests
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

## Fungsi preprocessing

In [79]:
def remove_emoji(text):
    return emoji.replace_emoji(text, replace='')

def cleaningText(text):
    text = remove_emoji(text)
    text = re.sub(r'@[A-Za-z0-9]+', '', text)
    text = re.sub(r'#[A-Za-z0-9]+', '', text)
    text = re.sub(r'RT[\s]', '', text)
    text = re.sub(r"http\S+", '', text)
    text = re.sub(r'[0-9]+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.replace('\n', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()
    return text

def casefoldingText(text): 
    return text.lower()

def load_slangwords_dict(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        slang_dict = json.load(file)
    return slang_dict

# Load slang words lokal (tidak perlu download lagi)
slangwords = load_slangwords_dict("combined_slang_words.txt")

def fix_slangwords(text):
    text = text.lower()
    words = text.split()
    fixed_words = [slangwords.get(word, word) for word in words]
    return ' '.join(fixed_words)

def tokenizingText(text): 
    return word_tokenize(text)

def filteringText(text): 
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords.update(stopwords.words('english'))
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    return [txt for txt in text if txt not in listStopwords]

def toSentence(list_words): 
    return ' '.join(list_words)

## Fungsi untuk prediksi sentimen

In [93]:
def predict_sentiment(text, model, tfidf_vectorizer):
    if not isinstance(text, str) or not text.strip():
        return "Input tidak valid"

    # Preprocessing
    text_clean = cleaningText(text)
    text_clean = casefoldingText(text_clean)
    text_clean = fix_slangwords(text_clean)
    text_tokens = tokenizingText(text_clean)
    text_tokens = filteringText(text_tokens)
    text_preprocessed = toSentence(text_tokens)

    # TF-IDF transform
    text_tfidf = tfidf_vectorizer.transform([text_preprocessed])

    # Prediksi
    prediction = model.predict(text_tfidf)
    predicted_label = prediction[0]  # langsung ambil label string

    return predicted_label

## Load model dan vectorizer

In [ ]:
logistic_regression = joblib.load('logistic_regression.pkl')
tfidf = joblib.load('tfidf_vectorizer.pkl')

## Input text baru

In [105]:
# Contoh kalimat
sample_text = "Aplikasi ini susah digunakan, jelek, ribet, sering error pokoknya aneh bikin kecewa!"
predicted_sentiment = predict_sentiment(sample_text, logistic_regression, tfidf)

if predicted_sentiment == 0:
    predicted_sentiment = "Negatif"
elif predicted_sentiment == 1:
    predicted_sentiment = "Netral"
else:
    predicted_sentiment = "Positif"

print(f"Prediksi Sentimen: {predicted_sentiment}")

Prediksi Sentimen: Negatif


In [106]:
# Contoh kalimat
sample_text = "Aplikasi ini mudah digunakan, aman, transfer gratis dan cepat, pokoknya bagus dan mantab "
predicted_sentiment = predict_sentiment(sample_text, logistic_regression, tfidf)

if predicted_sentiment == 0:
    predicted_sentiment = "Negatif"
elif predicted_sentiment == 1:
    predicted_sentiment = "Netral"
else:
    predicted_sentiment = "Positif"

print(f"Prediksi Sentimen: {predicted_sentiment}")

Prediksi Sentimen: Positif


In [108]:
# Contoh kalimat
sample_text = "Saya tidak bisa transfer!"
predicted_sentiment = predict_sentiment(sample_text, logistic_regression, tfidf)

if predicted_sentiment == 0:
    predicted_sentiment = "Negatif"
elif predicted_sentiment == 1:
    predicted_sentiment = "Netral"
else:
    predicted_sentiment = "Positif"

print(f"Prediksi Sentimen: {predicted_sentiment}")

Prediksi Sentimen: Netral
